# reroll 0.2.0: unconvertable-bucket failure analysis

This is a follow-up to `reroll_v0.1.0_failure_analysis.ipynb` and
`reroll_v0.1.1_failure_analysis.ipynb`, focused specifically on the
`unconvertable` category of `repodata_conversion.reroll_error` (wheels reroll
*tried* to convert but couldn't represent as a conda package -- as opposed to
`scope`, wheels deliberately out of scope, or `invalid`, malformed wheels).

**The bucket has shrunk 89.8%: 153,444 rows (v0.1.1) -> 15,595 rows (0.2.0).**
Small enough that this notebook loads it in full rather than sampling.

_(Summary and conclusion below are written after the analysis; see the final
section for the full breakdown and recommendations.)_

In [21]:
# Setup: connect to v.db (read-only) and load the FULL "unconvertable" bucket.
# Unlike the v0.1.0/v0.1.1 analyses (153k+ rows -- sampled/aggregated in SQL to
# stay cheap), reroll 0.2.0 has driven this bucket down to ~15,000 rows, small
# enough to load in its entirety and explore freely in-memory (per explicit
# override of the usual "don't table-scan v.db" guidance for this notebook).
import sqlite3
import re
import pandas as pd

DB_PATH = "/Users/anil/code/reroll-data/data/v.db"
con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
unconvertable_df = pd.read_sql_query(
    """
    SELECT project, filename, reroll_error
    FROM repodata_conversion
    WHERE reroll_error LIKE 'unconvertable:%'
    """,
    con,
)
con.close()
len(unconvertable_df), unconvertable_df.head()

(15595,
         project                              filename  \
 0  Activate-App  Activate_App-0.0.10-py3-none-any.whl   
 1  Activate-App   Activate_App-0.0.4-py3-none-any.whl   
 2  Activate-App   Activate_App-0.0.5-py3-none-any.whl   
 3  Activate-App   Activate_App-0.0.6-py3-none-any.whl   
 4  Activate-App   Activate_App-0.0.7-py3-none-any.whl   
 
                                                                                                                                                                                                                                                                                                                                                                                reroll_error  
 0  unconvertable: UnresolvedCondaNameError: no mapper resolved a conda name for 'pyqtchart': candidates=(Candidate(conda_name='pyqt', probability=0.0599, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmouth_relations'), Candidate(conda_na

In [22]:
# What exception types make up "unconvertable" errors (full population)?
exc_type = unconvertable_df["reroll_error"].str.extract(r"^unconvertable:\s*(\w+)")
unconvertable_df["exc_type"] = exc_type[0]
counts = unconvertable_df["exc_type"].value_counts()
pct = (counts / len(unconvertable_df) * 100).round(1)
pd.DataFrame({"count": counts, "pct_of_unconvertable": pct})

,count,pct_of_unconvertable
exc_type,,
UnconvertableRequirementError,7501,48.1
UnconvertableMarkerError,3736,24.0
UnresolvedCondaNameError,3002,19.2
InvalidCondaNameError,1277,8.2
UnconvertablePythonVersionEqualityError,79,0.5


In [23]:
# For each unconvertable exc_type: how many distinct projects (packages being
# rerolled) and distinct wheel filenames does it touch? Distinguishes "one huge
# project is spamming this bucket" from "broad-based, many projects affected".
by_type = unconvertable_df.groupby("exc_type").agg(
    n_rows=("reroll_error", "size"),
    n_projects=("project", "nunique"),
    n_filenames=("filename", "nunique"),
)
by_type["rows_per_project"] = (by_type["n_rows"] / by_type["n_projects"]).round(1)
by_type.sort_values("n_rows", ascending=False)

,n_rows,n_projects,n_filenames,rows_per_project
exc_type,,,,
UnconvertableRequirementError,7501,585,7501,12.8
UnconvertableMarkerError,3736,323,3736,11.6
UnresolvedCondaNameError,3002,117,3002,25.7
InvalidCondaNameError,1277,114,1277,11.2
UnconvertablePythonVersionEqualityError,79,14,79,5.6


In [24]:
# Look at one representative message per exc_type to understand each format
# before writing extraction regexes for sub-types.
pd.set_option("display.max_colwidth", 300)
for t, grp in unconvertable_df.groupby("exc_type"):
    print("=" * 80)
    print(t, f"(n={len(grp)})")
    print(grp["reroll_error"].iloc[0])
    print()

InvalidCondaNameError (n=1277)
unconvertable: InvalidCondaNameError: conda package name 'iptv-subscription-ip-tv-subscription-iptv-premium-subscription-worldwi
de-channels' exceeds 64 characters (81)

UnconvertableMarkerError (n=3736)
unconvertable: UnconvertableMarkerError: cannot convert marker OperatorNode(operator='and', _left=CompareNode(key='sys_platform'
, comparator='==', literal='darwin'), _right=CompareNode(key='platform_release', comparator='>', literal='20.6.0')): it still re
fers to unpermitted key(s) ['platform_release'] after evaluation

UnconvertablePythonVersionEqualityError (n=79)
unconvertable: UnconvertablePythonVersionEqualityError: python_version literal '2.5.2' is not major.minor precision, so '==' aga
inst it can never hold

UnconvertableRequirementError (n=7501)
unconvertable: UnconvertableRequirementError: cannot convert 'aleksis-app-resint==3.0.dev0+20220802190124.c3757b88': it has a lo
cal version label

UnresolvedCondaNameError (n=3002)
unconvertable: Unres

In [25]:
# UnconvertableRequirementError (48.1%, 7,501 rows -- now the single largest
# bucket): "cannot convert '<req>': <reason>". Cover both known message shapes
# from the v0.1.1 analysis up front, then check what's left unmatched.
req_errs = unconvertable_df[
    unconvertable_df["exc_type"] == "UnconvertableRequirementError"
].copy()


def classify_req_error(msg):
    m = re.search(r"cannot convert '[^']*':\s*(.+)$", msg)
    if m:
        return m.group(1)
    m = re.search(r"is not a legal conda extra name \(CEP-29\):\s*(.+)$", msg)
    if m:
        return f"illegal conda extra name: {m.group(1)}"
    return "OTHER/unrecognized"


reason = req_errs["reroll_error"].map(classify_req_error)
req_errs["reason"] = reason
template = reason.str.replace(r"'[^']*'", "<LIT>", regex=True).str.replace(
    r"\b\d+(?:\.\d+)*\b", "<NUM>", regex=True
)
req_errs["reason_template"] = template
counts = req_errs["reason_template"].value_counts()
pd.DataFrame({"count": counts, "pct": (counts / len(req_errs) * 100).round(1)})

,count,pct
reason_template,,
it has a local version label,6319,84.2
illegal conda extra name: must be <NUM>-<NUM> characters of [a-z0-9_.+-],672,9.0
OTHER/unrecognized,416,5.5
it has a direct URL reference,94,1.3


In [26]:
# 416 OTHER/unrecognized rows -- check the raw text to see if the v0.1.1
# "[when=...] selector matchspec generation bug" is still present, fixed, or
# changed shape.
unrecognized = req_errs[req_errs["reason"] == "OTHER/unrecognized"]
print(
    f"{len(unrecognized)} unrecognized rows, {unrecognized['project'].nunique()} distinct projects"
)
for s in unrecognized["reroll_error"].drop_duplicates().head(10):
    print(repr(s))
    print()

416 unrecognized rows, 9 distinct projects
"unconvertable: UnconvertableRequirementError: 'amulet-compiler-version ==3.0.0.309027957683952396889703.15.0.0.15000309', conve
rted from 'amulet-compiler-version==3.0.0.309027957683952396889703.15.0.0.15000309', is not a valid matchspec"

"unconvertable: UnconvertableRequirementError: 'amulet-compiler-version ==4.309027957683952396889703.17', converted from 'amulet
-compiler-version==4.309027957683952396889703.17', is not a valid matchspec"

"unconvertable: UnconvertableRequirementError: 'amulet-compiler-version ==1.3.0.309027957683952396889703.15.0.0.15000309', conve
rted from 'amulet-compiler-version==1.3.0.309027957683952396889703.15.0.0.15000309', is not a valid matchspec"

"unconvertable: UnconvertableRequirementError: 'amulet-compiler-version ==3.0.0.309027957683952396889703.15.0.0.15000100', conve
rted from 'amulet-compiler-version==3.0.0.309027957683952396889703.15.0.0.15000100', is not a valid matchspec"

"unconvertable: Unconvertab

In [27]:
# Confirm: is the v0.1.1 "[when=...] selector produces an invalid matchspec"
# bug still present at all in 0.2.0, or is amulet-compiler-version's
# numeric-overflow version string now the *only* thing left in this sub-bucket?
when_bug = unrecognized[
    unrecognized["reroll_error"].str.contains(r"\[when=", regex=False)
]
print(f"rows still hitting the [when=...] bug: {len(when_bug)}")
print()
print("all 9 distinct projects behind the 416 remaining OTHER/unrecognized rows:")
print(unrecognized["project"].value_counts())

rows still hitting the [when=...] bug: 0

all 9 distinct projects behind the 416 remaining OTHER/unrecognized rows:
project
amulet-nbt              82
amulet-zlib             61
amulet-leveldb          60
amulet-core             51
amulet-utils            51
amulet-game             35
amulet-anvil            28
amulet-resource-pack    27
amulet-level            21
Name: count, dtype: int64


In [28]:
# "it has a local version label" (84.2%, 6,319 rows) is now the single biggest
# sub-bucket in the whole unconvertable population. Local version labels
# (PEP 440 `+something` suffixes, e.g. `1.0+cpu`, `2.1.0+cu121`) are not
# expressible in conda's version spec -- structurally unconvertable, not a bug.
# Check concentration: is this a handful of projects (e.g. pytorch/cu variants
# depending on each other) or broad-based?
local_ver = req_errs[req_errs["reason_template"] == "it has a local version label"]
n_projects = local_ver["project"].nunique()
print(f"{len(local_ver):,} rows from {n_projects:,} distinct projects")
print(f"-> average {len(local_ver) / n_projects:.1f} rows/project")
print()
top_projects = local_ver["project"].value_counts()
print(top_projects.head(20))
print()
cum_share = top_projects.cumsum() / len(local_ver) * 100
for n in [5, 10, 25, 50, 100]:
    print(
        f"  top {n:>4} projects -> {cum_share.iloc[min(n, len(cum_share)) - 1]:.1f}% of local-version-label rows"
    )

6,319 rows from 436 distinct projects
-> average 14.5 rows/project

project
ipex-llm               1152
bigdl-nano              808
bigdl-llm               397
apache-beam             340
pruna-pro               179
torch-npu               173
rclip                   160
vllm-cpu                144
determined              133
vllm-cpu-avx512bf16     121
vllm-cpu-avx512vnni     121
vllm-cpu-amxbf16        118
xinference               98
rerun-sdk                96
vllm-cpu-avx512          94
marin-core               77
torch-rbln               76
MindsDB                  67
rara-digitizer           67
ultimate-rvc             63
Name: count, dtype: int64

  top    5 projects -> 45.5% of local-version-label rows
  top   10 projects -> 57.1% of local-version-label rows
  top   25 projects -> 75.0% of local-version-label rows
  top   50 projects -> 84.6% of local-version-label rows
  top  100 projects -> 91.0% of local-version-label rows


In [29]:
# Look at the actual dependency strings behind the top offenders -- confirm
# these are genuinely PEP 440 local versions (hardware/build variant tags),
# not something reroll could plausibly strip and still convert correctly.
sample = unconvertable_df.loc[local_ver.index].merge(
    local_ver[["reason"]], left_index=True, right_index=True
)
extracted = sample["reroll_error"].str.extract(r"cannot convert '([^']*)'")
sample["requirement"] = extracted[0]
for proj in ["ipex-llm", "bigdl-nano", "apache-beam", "vllm-cpu"]:
    print(proj + ":")
    print(
        sample.loc[sample["project"] == proj, "requirement"]
        .drop_duplicates()
        .head(5)
        .to_list()
    )
    print()

ipex-llm:
['intel-extension-for-pytorch==2.1.10+xpu', 'intel-extension-for-pytorch==2.0.110+xpu', 'torch==2.1.2+cpu', 'torch==2.6.0+xpu',
'torch==2.3.1+cxx11.abi']

bigdl-nano:
['torch==1.13.0a0+git6c9b55e']

apache-beam:
['torch==2.8.0+cpu']

vllm-cpu:
['torch==2.6.0+cpu', 'torch==2.8.0+cpu', 'torch==2.9.1+cpu', 'torch==2.10.0+cpu', 'torch==2.11.0+cpu']


In [30]:
# UnconvertableMarkerError (24.0%, 3,736 rows). Two distinct message shapes:
#   1. "cannot convert marker ...: it still refers to unpermitted key(s) [...]"
#      (markers on keys reroll doesn't evaluate/permit at all, e.g. platform_release)
#   2. "cannot convert the marker in '<req>' to a matchspec: <reason>"
#      (markers on permitted keys like python_version, but with an unsupported
#      literal/comparator/operator shape)
# Collapse into a small number of top-level buckets (the raw templates explode
# into dozens of near-duplicates that only differ by how many values are
# listed in an `in`/`not in` set).
marker_errs = unconvertable_df[
    unconvertable_df["exc_type"] == "UnconvertableMarkerError"
].copy()
reason = marker_errs["reroll_error"].str.extract(r"UnconvertableMarkerError:\s*(.+)$")[
    0
]
marker_errs["reason"] = reason
unpermitted = reason.str.extract(r"unpermitted key\(s\) \['([^']*)'")[0]


def bucket_marker(msg, key):
    if pd.notna(key):
        return f"unpermitted key: {key}"
    if "markers are not supported" in msg:
        return "'in'/'not in' marker unsupported"
    if "not a valid version" in msg:
        return "python_version literal not a valid version (e.g. '2.7.*', '3.x', 'dev')"
    if "is not supported for" in msg:
        return "unsupported comparator (e.g. '~=') for python_version(_full)"
    if "not a plain major.minor" in msg:
        return "python_version literal not major.minor(.micro) precision"
    return "OTHER/unrecognized"


marker_errs["bucket"] = [bucket_marker(m, k) for m, k in zip(reason, unpermitted)]
counts = marker_errs["bucket"].value_counts()
pd.DataFrame({"count": counts, "pct": (counts / len(marker_errs) * 100).round(1)})

,count,pct
bucket,,
'in'/'not in' marker unsupported,1872,50.1
"python_version literal not a valid version (e.g. '2.7.*', '3.x', 'dev')",610,16.3
unsupported comparator (e.g. '~=') for python_version(_full),532,14.2
unpermitted key: platform_release,375,10.0
python_version literal not major.minor(.micro) precision,239,6.4
unpermitted key: platform_version,84,2.2
unpermitted key: platform_machine,13,0.3
unpermitted key: sys_platform,8,0.2
unpermitted key: platform_python_implementation,3,0.1


In [31]:
# "'in'/'not in' marker unsupported" (50.1%, 1,872 rows) -- reroll's marker
# evaluator can't express PEP 508 `python_version in "3.7, 3.8, 3.9"` style
# set-membership tests as a conda matchspec at all (fundamentally different
# grammar, not obviously fixable the way a comparator bug would be). Check
# how concentrated this is, and see a few real requirement strings.
in_bug = marker_errs[marker_errs["bucket"] == "'in'/'not in' marker unsupported"]
print(f"{len(in_bug)} rows, {in_bug['project'].nunique()} distinct projects")
print(in_bug["project"].value_counts().head(10))
print()
sample_idx = in_bug.index[:5]
for s in unconvertable_df.loc[sample_idx, "reroll_error"]:
    m = re.search(r"cannot convert the marker in '([^']*)'", s)
    print(m.group(1) if m else s)

1872 rows, 167 distinct projects
project
an-website            115
codeforlife-portal     76
codeforlife            73
rapid-router           72
cfl-common             67
ezbeq                  62
basking-sdk            60
viur-core              57
declafe                56
PlexTraktSync          55
Name: count, dtype: int64

httplib2==0.21.0; python_version >= "2.7" and python_version not in "3.0, 3.1, 3.2, 3.3"
httplib2==0.21.0; python_version >= "2.7" and python_version not in "3.0, 3.1, 3.2, 3.3"
future==0.18.2; python_version >= "2.6" and python_version not in "3.0, 3.1, 3.2"
future==0.18.2; python_version >= "2.6" and python_version not in "3.0, 3.1, 3.2"
future==0.18.2; python_version >= "2.6" and python_version not in "3.0, 3.1, 3.2"


In [38]:
# Concrete examples: "unsupported comparator ('~=') for python_version(_full)"
# (3.4%, 532 rows). `~=` is PEP 440's "compatible release" operator (e.g.
# `~= 2.7` means `>= 2.7, == 2.*`) -- reroll's marker evaluator only knows how
# to translate `==`, `!=`, `<`, `<=`, `>`, `>=` against python_version(_full)
# into a matchspec, not `~=`.
comparator_bug = marker_errs[
    marker_errs["bucket"]
    == "unsupported comparator (e.g. '~=') for python_version(_full)"
]
print(
    f"{len(comparator_bug)} rows, {comparator_bug['project'].nunique()} distinct projects"
)
print()
print("top offending projects:")
print(comparator_bug["project"].value_counts().head(10))
print()

sample = unconvertable_df.loc[comparator_bug.index]
extracted = sample["reroll_error"].str.extract(
    r"cannot convert the marker in '([^']*)'"
)
sample = sample.assign(requirement=extracted[0])
print("sample requirement strings that triggered this:")
for _, row in sample.drop_duplicates(subset="requirement").head(10).iterrows():
    print(f"  {row['project']!r:30}  {row['requirement']}")

532 rows, 37 distinct projects

top offending projects:
project
strawberry-graphql           113
pyserde                       61
kapitan                       57
nicegui                       57
psevencore                    46
mikro-next                    32
snowflake-snowpark-python     21
pike-smb2                     16
pytest-bdd-ng                 15
curlipie                      13
Name: count, dtype: int64

sample requirement strings that triggered this:
  'PyAthena'                      futures; python_version ~= "2.7"
  'aiohttp-asgi-connector'        aiohttp<3.11,>=3.1.0; python_version ~= "3.8.0"
  'autosysloguru'                 pathlib2; python_version ~= "2.7"
  'awswrangler'                   pandas<1.2.0,>=1.1.0; python_full_version ~= "3.6.2"
  'awswrangler'                   numpy<1.19.0,>=1.18.0; python_full_version ~= "3.6.2"
  'cartographer3d-plugin'         numpy~=1.16; python_version ~= "3.10"
  'clickzetta-zettapark-python'   cloudpickle==2.2.1; python_versio

In [32]:
# UnresolvedCondaNameError (19.2%, 3,002 rows) -- down massively from 86,859
# (56.6%) in v0.1.1. Which dependency names still can't be resolved, and do
# any of them have near-miss candidates (i.e. a mapper fix, not a "genuinely
# unknown package", would close the gap)?
unres = unconvertable_df[
    unconvertable_df["exc_type"] == "UnresolvedCondaNameError"
].copy()
dep_name = unres["reroll_error"].str.extract(
    r"no mapper resolved a conda name for '([^']*)'"
)[0]
unres["dep_name"] = dep_name
has_candidates = unres["reroll_error"].str.contains(
    r"candidates=\(Candidate", regex=True
)
unres["has_candidates"] = has_candidates

print(f"distinct unresolved dependency names: {unres['dep_name'].nunique()}")
print()
print("top unresolved dependency names by row count:")
top_names = unres["dep_name"].value_counts()
print(top_names.head(15))
print()
print(
    f"has >=1 candidate but still unresolved: {has_candidates.sum()} / {len(unres)} ({has_candidates.mean() * 100:.1f}%)"
)

distinct unresolved dependency names: 6

top unresolved dependency names by row count:
dep_name
functools32     2412
pyqtchart        556
alchemiscale      17
oasis              7
nr-config          5
sktools            5
Name: count, dtype: int64

has >=1 candidate but still unresolved: 3002 / 3002 (100.0%)


In [33]:
# onnxruntime/modal (the v0.1.1 top offenders, 49.3% of that release's whole
# unconvertable population) are GONE from this list entirely -- the mapper fix
# worked. functools32 alone is now 80.3% of this bucket. Look at its candidates
# (100% of rows have >=1 candidate) to see whether it's a near-miss confidence
# threshold issue or genuinely has no good match.
pd.set_option("display.max_colwidth", 400)
for name in ["functools32", "pyqtchart", "alchemiscale"]:
    print("=" * 80)
    print(name)
    print(unres.loc[unres["dep_name"] == name, "reroll_error"].iloc[0])
    print()

functools32
unconvertable: UnresolvedCondaNameError: no mapper resolved a conda name for 'functools32': candidates=(Candidate(conda_name='fu
nctools32', probability=0.4275, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmouth_relations'), Candidate(
conda_name='jsonschema', probability=0.0086, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmouth_relations'
))

pyqtchart
unconvertable: UnresolvedCondaNameError: no mapper resolved a conda name for 'pyqtchart': candidates=(Candidate(conda_name='pyqt
', probability=0.0599, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmouth_relations'), Candidate(conda_nam
e='pyqtchart', probability=0.6017, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmouth_relations'))

alchemiscale
unconvertable: UnresolvedCondaNameError: no mapper resolved a conda name for 'alchemiscale': candidates=(Candidate(conda_name='a
lchemiscale-client', probability=0.57, source=<CandidateSource.PA

In [34]:
# Side note: PythonRangeMismatchError was 23.2% (35,549 rows) of the v0.1.1
# unconvertable population but is completely absent from the 0.2.0 exc_type
# breakdown above. Checking reroll's source (errors.py) explains why: it's
# defined as a `RerollInvalidWheelError` leaf ("the same shape as
# MetadataFilenameMismatchError... No valid record can be emitted for it"),
# which logs under the "invalid" category, not "unconvertable". Confirm with
# a cheap aggregate query whether it was reclassified (still present, just
# under a different reroll_error prefix) or genuinely eliminated.
con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
counts_by_prefix = pd.read_sql_query(
    """
    SELECT
        CASE
            WHEN reroll_error LIKE 'scope:%' THEN 'scope'
            WHEN reroll_error LIKE 'invalid:%' THEN 'invalid'
            WHEN reroll_error LIKE 'unconvertable:%' THEN 'unconvertable'
            WHEN reroll_error LIKE 'unexpected:%' THEN 'unexpected'
            ELSE 'other/no-error'
        END AS category,
        SUM(CASE WHEN reroll_error LIKE '%PythonRangeMismatchError%' THEN 1 ELSE 0 END) AS python_range_mismatch_rows,
        COUNT(*) AS total_rows
    FROM repodata_conversion
    GROUP BY category
    """,
    con,
)
con.close()
counts_by_prefix

,category,python_range_mismatch_rows,total_rows
0,invalid,29861,335241
1,other/no-error,0,10422984
2,scope,0,1347989
3,unconvertable,0,15595
4,unexpected,0,45


In [35]:
# InvalidCondaNameError (8.2%, 1,277 rows) -- exactly the same count as
# v0.1.1's InvalidCondaNameError total (1,275 long-name + 2 homoglyph rows).
# Confirm the message shapes are unchanged (this bucket wasn't a 0.2.0 target).
invalid_errs = unconvertable_df[unconvertable_df["exc_type"] == "InvalidCondaNameError"]
shape = invalid_errs["reroll_error"].str.extract(r"InvalidCondaNameError:\s*(.+?)\s*'")[
    0
]
print(shape.value_counts())
print()
long_names = (
    invalid_errs["reroll_error"]
    .str.extract(r"conda package name '([^']*)' exceeds")[0]
    .dropna()
)
print(
    f"{long_names.nunique()} distinct over-length names across {invalid_errs['project'].nunique()} projects"
)
weird = invalid_errs[
    ~invalid_errs["reroll_error"].str.contains("exceeds 64 characters")
]
print(f"non-length rows: {len(weird)}")
print(weird[["project", "reroll_error"]].to_string())

0
conda package name    1275
'tааmo                   1
'tаааааmo                1
Name: count, dtype: int64

113 distinct over-length names across 114 projects
non-length rows: 2
      project                                                                                 reroll_error
12148    t-mo     unconvertable: InvalidCondaNameError: 'tааmo' is not a legal conda package name (CEP 26)
12149    t-mo  unconvertable: InvalidCondaNameError: 'tаааааmo' is not a legal conda package name (CEP 26)


In [36]:
# UnconvertablePythonVersionEqualityError (0.5%, 79 rows) -- a brand-new leaf
# type in 0.2.0 (per reroll's errors.py docstring: split out of
# UnconvertableMarkerError "so real-world frequency can be measured before
# deciding how/whether to represent it"). `python_version == "X.Y.Z"` with a
# nonzero micro segment can never hold (python_version is always major.minor),
# so this is a constant-false/true marker with no matchspec equivalent.
pyver_eq = unconvertable_df[
    unconvertable_df["exc_type"] == "UnconvertablePythonVersionEqualityError"
].copy()
print(f"{len(pyver_eq)} rows, {pyver_eq['project'].nunique()} distinct projects")
print()
print(pyver_eq["project"].value_counts())
print()
lit = pyver_eq["reroll_error"].str.extract(r"literal '([^']*)'")[0]
print("literal python_version values involved:")
print(lit.value_counts())

79 rows, 14 distinct projects

project
wetterdienst            22
voxrow                  15
cmd2                    11
compbiolab-CLI           5
rpaframework-core        5
rpaframework-windows     5
cmdtube                  3
congressgov              3
streamlit-canary         3
alienGame                2
peaks2utr                2
hawkdata                 1
siliconcompiler          1
tvc-demo-flask           1
Name: count, dtype: int64

literal python_version values involved:
0
3.9.7     26
3.7.13    15
3.5.2     11
3.8.1     10
3.7.6      5
11.0.2     3
3.14.1     3
2.5.2      2
3.8.5      2
1.2.2      1
3.9.6      1
Name: count, dtype: int64


In [37]:
# Roll everything up: one table, sub-type counts as % of the FULL 15,595-row
# 0.2.0 unconvertable population, for the write-up below.
total = len(unconvertable_df)

req_sub = req_errs["reason_template"].value_counts()
marker_sub = marker_errs["bucket"].value_counts()
unres_sub = unres["dep_name"].value_counts()

rows = [
    (
        "UnconvertableRequirementError",
        "PEP 440 local version label (torch/ipex-llm/vllm-cpu hardware variants)",
        req_sub["it has a local version label"],
    ),
    (
        "UnconvertableRequirementError",
        "illegal conda extra name (CEP-29)",
        req_sub[
            "illegal conda extra name: must be <NUM>-<NUM> characters of [a-z0-9_.+-]"
        ],
    ),
    (
        "UnconvertableRequirementError",
        "amulet-compiler-version numeric-overflow version",
        req_sub["OTHER/unrecognized"],
    ),
    (
        "UnconvertableRequirementError",
        "direct URL reference",
        req_sub["it has a direct URL reference"],
    ),
    (
        "UnconvertableMarkerError",
        "'in'/'not in' marker unsupported",
        marker_sub["'in'/'not in' marker unsupported"],
    ),
    (
        "UnconvertableMarkerError",
        "python_version literal not a valid version",
        marker_sub[
            "python_version literal not a valid version (e.g. '2.7.*', '3.x', 'dev')"
        ],
    ),
    (
        "UnconvertableMarkerError",
        "unsupported comparator (e.g. '~=')",
        marker_sub["unsupported comparator (e.g. '~=') for python_version(_full)"],
    ),
    (
        "UnconvertableMarkerError",
        "unpermitted key: platform_release/_version/_machine/etc.",
        marker_sub[
            [k for k in marker_sub.index if k.startswith("unpermitted key")]
        ].sum(),
    ),
    (
        "UnconvertableMarkerError",
        "python_version literal not major.minor precision",
        marker_sub["python_version literal not major.minor(.micro) precision"],
    ),
    ("UnresolvedCondaNameError", "functools32 unresolved", unres_sub["functools32"]),
    ("UnresolvedCondaNameError", "pyqtchart unresolved", unres_sub["pyqtchart"]),
    (
        "UnresolvedCondaNameError",
        "other 4 dep names unresolved",
        unres_sub.drop(["functools32", "pyqtchart"]).sum(),
    ),
    ("InvalidCondaNameError", "name >64 chars (SEO-spam project names)", 1275),
    ("InvalidCondaNameError", "homoglyph/Cyrillic lookalike name", 2),
    (
        "UnconvertablePythonVersionEqualityError",
        "python_version ==/!= a non-major.minor literal",
        79,
    ),
]
summary_all = pd.DataFrame(rows, columns=["exc_type", "sub_bucket", "n"])
summary_all["pct_of_all_unconvertable"] = (summary_all["n"] / total * 100).round(1)
assert summary_all["n"].sum() == total, (summary_all["n"].sum(), total)
summary_all.sort_values("n", ascending=False)

,exc_type,sub_bucket,n,pct_of_all_unconvertable
0,UnconvertableRequirementError,PEP 440 local version label (torch/ipex-llm/vllm-cpu hardware variants),6319,40.5
9,UnresolvedCondaNameError,functools32 unresolved,2412,15.5
4,UnconvertableMarkerError,'in'/'not in' marker unsupported,1872,12.0
12,InvalidCondaNameError,name >64 chars (SEO-spam project names),1275,8.2
1,UnconvertableRequirementError,illegal conda extra name (CEP-29),672,4.3
5,UnconvertableMarkerError,python_version literal not a valid version,610,3.9
10,UnresolvedCondaNameError,pyqtchart unresolved,556,3.6
6,UnconvertableMarkerError,unsupported comparator (e.g. '~='),532,3.4
7,UnconvertableMarkerError,unpermitted key: platform_release/_version/_machine/etc.,483,3.1
2,UnconvertableRequirementError,amulet-compiler-version numeric-overflow version,416,2.7


## Conclusion: what changed since v0.1.1, and what work remains

### The bucket shrank 89.8% (153,444 -> 15,595 rows), driven by one confirmed bug fix

`git log` on `reroll` shows commit `b9ab3ef` ("V0.1.0 bug fixes (#38)") landed
*exactly* the fix v0.1.0's report flagged as priority #1:
`aggregator_mapper`'s `if source is PARSELMOUTH: ... else: ...` structure used
to make the "sole mapper's candidate >= 0.9 probability" rule and "parselmouth's
only candidate" rule mutually exclusive, so any single-mapper parselmouth
result with *more than one* candidate was rejected regardless of confidence.
That one-line reorder is why `onnxruntime` (58,824 rows, 38.3% of v0.1.1's
*entire* unconvertable population) and `modal` (10,758 rows, 7.0%) -- v0.1.1's
#1 and #2 offenders -- are **completely absent** from 0.2.0's `UnresolvedCondaNameError`
bucket. The same release also fully fixed the `[when=...]` selector matchspec-generation
bug (2,232 rows in v0.1.1; 0 rows remain) and evidently the `PythonRangeMismatchError`
false-positive intersection bug (5,898 rows in v0.1.1) -- though that error was
also reclassified from `unconvertable` to `invalid` in the process (it's a
wheel/metadata contradiction, not an unconvertable dependency shape), so it no
longer appears in this bucket at all; 29,861 rows remain under `invalid:%PythonRangeMismatchError%`.

### Breakdown of the 15,595 rows that remain

| exc_type | sub-bucket | n | % of all unconvertable | actionable? |
|---|---|--:|--:|---|
| `UnconvertableRequirementError` | PEP 440 local version label (torch/ipex-llm/vllm-cpu `+cpu`/`+xpu` hardware variants) | 6,319 | 40.5% | No -- structural (conda has no local-version equivalent) |
| `UnresolvedCondaNameError` | `functools32` unresolved (best candidate = exact name, prob. 0.4275) | 2,412 | 15.5% | **Maybe** -- see below |
| `UnconvertableMarkerError` | `in`/`not in` marker unsupported | 1,872 | 12.0% | No -- structural (no matchspec equivalent to set membership) |
| `InvalidCondaNameError` | name >64 chars (SEO-spam PyPI names) | 1,275 | 8.2% | No -- structural (CEP 26 length limit); unchanged since v0.1.1 |
| `UnconvertableRequirementError` | illegal conda extra name (CEP-29) | 672 | 4.3% | No -- structural |
| `UnconvertableMarkerError` | `python_version` literal not a valid version (`'2.7.*'`, `'3.x'`, `'dev'`) | 610 | 3.9% | No -- malformed upstream metadata |
| `UnresolvedCondaNameError` | `pyqtchart` unresolved (best candidate = exact name, prob. 0.6017) | 556 | 3.6% | **Maybe** -- see below |
| `UnconvertableMarkerError` | unsupported comparator (`~=`) for `python_version(_full)` | 532 | 3.4% | **Yes** -- see below |
| `UnconvertableMarkerError` | unpermitted key (`platform_release`/`_version`/`_machine`/`sys_platform`/`platform_python_implementation`) | 483 | 3.1% | No -- structural (free-form OS-release strings, no finite subdir split) |
| `UnconvertableRequirementError` | `amulet-compiler-version` numeric-overflow version | 416 | 2.7% | No -- upstream package publishes absurd version numbers |
| `UnconvertableMarkerError` | `python_version` literal not major.minor precision | 239 | 1.5% | No -- structural |
| `UnconvertableRequirementError` | direct URL reference | 94 | 0.6% | No -- structural |
| `UnconvertablePythonVersionEqualityError` | `python_version ==`/`!=` a non-major.minor literal | 79 | 0.5% | No -- structural (constant-false/true marker; new leaf type in 0.2.0) |
| `UnresolvedCondaNameError` | 4 other dep names (`alchemiscale`, `oasis`, `nr-config`, `sktools`) | 34 | 0.2% | No -- genuinely ambiguous/unmapped |
| `InvalidCondaNameError` | homoglyph/Cyrillic lookalike name (`tааmo`) | 2 | 0.0% | No -- unchanged since v0.1.1 |

**77.6%** of what remains (everything except the three rows flagged
actionable above) is `reroll` correctly refusing to convert something with no
valid conda representation -- PEP 440/508 constructs (local versions,
set-membership markers, direct URLs, non-major-minor equality) that simply
don't have a matchspec equivalent, plus upstream data problems (SEO-spam/
homoglyph names, one package with intentionally absurd version numbers).
None of that is a bug. The remaining **22.4%** (3,500 rows) breaks down into
one confirmed, cheap fix and one open question -- both below.

### Confirmed actionable fix: `~=` comparator on `python_version(_full)` (532 rows, 3.4%)

PEP 440's "compatible release" operator, `~= X.Y[.Z]`, expands to a plain
range: `~= X.Y` means `>=X.Y,<(X+1).0`, and `~= X.Y.Z` means
`>=X.Y.Z,<X.(Y+1).0`. reroll's marker evaluator already converts `==`, `!=`,
`<`, `<=`, `>`, `>=` against `python_version`/`python_full_version` into a
matchspec fragment -- `~=` is simply missing from that comparator table, not
a semantic gap. Rewriting `~=` to the equivalent `>=`/`<` pair before handing
off to the existing conversion logic would close all 532 rows losslessly, no
information lost and no ambiguity to resolve. Broad-based (37 distinct
projects; `strawberry-graphql`, `pyserde`, `kapitan`, `nicegui` are the top
few, none dominating), e.g.:

```
awswrangler: pandas<1.2.0,>=1.1.0; python_full_version ~= "3.6.2"
  -> unconvertable: UnconvertableMarkerError: cannot convert the marker in
     'pandas<1.2.0,>=1.1.0; python_full_version ~= "3.6.2"' to a matchspec:
     comparator '~=' is not supported for python_full_version
```

This is the highest-value item in the whole bucket: unlike every other row
here, it's both non-trivial in size *and* has a concrete, mechanical fix with
no design tradeoff.

### Open question: `functools32` + `pyqtchart` (2,968 rows, 19.0%)

Both have a single mapper (`parselmouth_relations`) proposing >1 candidate,
where the *top* candidate's `conda_name` is the exact same string as the
input name -- `functools32` at `probability=0.4275`, `pyqtchart` at
`probability=0.6017` -- but both fall well short of the `>= 0.9` bar
`aggregator_mapper` now applies uniformly to multi-candidate single-mapper
results (`reroll/name_mapping.py:105-110`). Unlike the `onnxruntime`/`modal`
cases this fixed, these aren't high-confidence results being wrongly
discarded -- parselmouth itself is only ~43-60% confident. Worth a follow-up
question for whoever owns the mapper: should an exact-string-match candidate
(regardless of source confidence) get a lower bar than an arbitrary rename
suggestion? If so, these ~3,000 rows would resolve; if the low confidence is
legitimate (e.g. `functools32` is a Python-2-only backport that may not
actually exist as a conda package), `UnresolvedCondaNameError` is the correct
outcome and no fix is needed.

### Recommendation

One concrete, low-risk fix is worth doing before/for a future release:
**add `~=` to the marker-comparator conversion table** (rewrite it to the
equivalent `>=`/`<` pair), which closes 532 rows (3.4%) with no design
tradeoffs. Beyond that, the only other item worth triaging is whether
`functools32`/`pyqtchart` should resolve despite their sub-0.9 confidence
(2,968 rows, 19.0%) -- a mapper-policy question, not a bug fix. Everything
else in this bucket (77.6%) is `reroll` behaving correctly against
structurally-unconvertable PEP 440/508 constructs.